# PyTorch: Guía Completa para Deep Learning

## Introducción

**PyTorch** es una biblioteca de Deep Learning de código abierto desarrollada principalmente por Facebook AI Research (FAIR). Se ha convertido en el estándar de la industria y la academia debido a su flexibilidad, facilidad de uso y potente sistema de diferenciación automática.

### ¿Por qué PyTorch?

1. **Pythonic**: PyTorch se siente natural para programadores de Python. No hay "magia" oculta; el código hace lo que parece.

2. **Grafos dinámicos**: A diferencia de TensorFlow 1.x, PyTorch construye el grafo computacional sobre la marcha, facilitando el debugging y permitiendo arquitecturas dinámicas.

3. **Comunidad activa**: Amplia documentación, tutoriales, y soporte de la comunidad.

4. **Ecosistema rico**: Torchvision (imágenes), Torchaudio (audio), Torchtext (NLP), y muchas más.

### Objetivos de este notebook

Al finalizar, serás capaz de:
- Crear y manipular tensores
- Entender el sistema de autograd (diferenciación automática)
- Construir redes neuronales con `torch.nn`
- Entrenar modelos con optimizadores
- Gestionar datos con DataLoader

### Contenido

1. [Tensores: La Estructura Fundamental](#1.-Tensores:-La-Estructura-Fundamental)
2. [Operaciones con Tensores](#2.-Operaciones-con-Tensores)
3. [Autograd: Diferenciación Automática](#3.-Autograd:-Diferenciación-Automática)
4. [torch.nn: Construcción de Redes](#4.-torch.nn:-Construcción-de-Redes)
5. [torch.optim: Optimizadores](#5.-torch.optim:-Optimizadores)
6. [DataLoader: Gestión de Datos](#6.-DataLoader:-Gestión-de-Datos)
7. [Ciclo de Entrenamiento Completo](#7.-Ciclo-de-Entrenamiento-Completo)
8. [Referencia de Funciones](#8.-Referencia-de-Funciones)

---

In [ ]:
# Importaciones necesarias
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

# Verificar versión e instalación
print(f"PyTorch versión: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

---

## 1. Tensores: La Estructura Fundamental

### ¿Qué es un Tensor?

Un **tensor** es una generalización de escalares, vectores y matrices a dimensiones arbitrarias:

| Dimensiones | Nombre | Ejemplo |
|-------------|--------|--------|
| 0D | Escalar | Un número: `5` |
| 1D | Vector | Una lista: `[1, 2, 3]` |
| 2D | Matriz | Una tabla: `[[1,2], [3,4]]` |
| 3D | Tensor 3D | Una imagen RGB: `(3, 224, 224)` |
| 4D | Tensor 4D | Batch de imágenes: `(32, 3, 224, 224)` |

En PyTorch, los tensores son similares a los arrays de NumPy, pero con dos diferencias cruciales:

1. **Pueden ejecutarse en GPU** para cálculos acelerados
2. **Soportan diferenciación automática** (autograd)

### Convenciones de dimensiones en Deep Learning

Es importante entender cómo se organizan los datos:

- **Datos tabulares**: `(batch_size, features)` → `(32, 10)` = 32 muestras, 10 features
- **Imágenes**: `(batch_size, channels, height, width)` → `(32, 3, 224, 224)`
- **Secuencias**: `(batch_size, sequence_length, features)` → `(32, 100, 256)`

La primera dimensión casi siempre es el **batch_size** (número de muestras procesadas simultáneamente).

### 1.1 Creación de Tensores

PyTorch ofrece múltiples formas de crear tensores. A continuación, se documentan las funciones principales:

In [ ]:
print("="*70)
print("CREACIÓN DE TENSORES")
print("="*70)

# ─────────────────────────────────────────────────────────────────────
# torch.tensor(data, dtype=None, device=None, requires_grad=False)
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Crea un tensor a partir de datos existentes (lista, tupla, array).
#   Siempre COPIA los datos (no comparte memoria).
#
# PARÁMETROS:
#   data: Los datos a convertir en tensor
#   dtype: Tipo de dato (torch.float32, torch.int64, etc.)
#   device: Dispositivo ('cpu', 'cuda', 'cuda:0')
#   requires_grad: Si True, PyTorch rastreará operaciones para gradientes
#
# RETORNA:
#   Un nuevo tensor con los datos proporcionados

print("\n1. torch.tensor() - Desde datos existentes")
print("-" * 50)

# Desde lista de Python
t1 = torch.tensor([1, 2, 3, 4])
print(f"Desde lista:        {t1}")
print(f"  dtype:            {t1.dtype}")

# Especificando tipo
t2 = torch.tensor([1, 2, 3], dtype=torch.float32)
print(f"Con float32:        {t2}")

# Matriz (lista de listas)
t3 = torch.tensor([[1, 2], [3, 4], [5, 6]])
print(f"Matriz 3x2:         shape {t3.shape}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# torch.from_numpy(ndarray)
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Crea un tensor que COMPARTE MEMORIA con el array NumPy.
#   ¡CUIDADO! Modificar uno afecta al otro.
#
# PARÁMETROS:
#   ndarray: Array de NumPy
#
# RETORNA:
#   Tensor que comparte datos con el array original
#
# NOTA:
#   El tensor resultante no puede tener requires_grad=True.
#   Para eso, usar .clone() o torch.tensor().

print("\n2. torch.from_numpy() - Compartir memoria con NumPy")
print("-" * 50)

np_array = np.array([1.0, 2.0, 3.0])
t_numpy = torch.from_numpy(np_array)

print(f"Array NumPy:        {np_array}")
print(f"Tensor:             {t_numpy}")

# Demostrar que comparten memoria
np_array[0] = 999
print(f"\nDespués de modificar NumPy:")
print(f"Array NumPy:        {np_array}")
print(f"Tensor (afectado):  {t_numpy}  ← ¡También cambió!")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# torch.zeros(size, dtype=None, device=None, requires_grad=False)
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Crea un tensor lleno de ceros.
#
# PARÁMETROS:
#   size: Tupla o enteros indicando las dimensiones
#
# USO TÍPICO:
#   Inicializar buffers, crear máscaras, padding.

print("\n3. torch.zeros() - Tensor de ceros")
print("-" * 50)

zeros_1d = torch.zeros(5)
zeros_2d = torch.zeros(2, 3)  # También: torch.zeros((2, 3))
zeros_3d = torch.zeros(2, 3, 4)

print(f"1D (5,):            {zeros_1d}")
print(f"2D (2,3):           shape {zeros_2d.shape}")
print(f"3D (2,3,4):         shape {zeros_3d.shape}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# torch.ones(size, dtype=None, device=None, requires_grad=False)
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Crea un tensor lleno de unos.
#
# USO TÍPICO:
#   Máscaras de atención, inicializaciones, broadcasting.

print("\n4. torch.ones() - Tensor de unos")
print("-" * 50)

ones = torch.ones(3, 3)
print(f"Matriz 3x3 de unos:\n{ones}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# torch.rand(size) / torch.randn(size)
# ─────────────────────────────────────────────────────────────────────
# torch.rand():
#   Distribución UNIFORME en el intervalo [0, 1)
#   Todos los valores entre 0 y 1 son igualmente probables.
#
# torch.randn():
#   Distribución NORMAL (Gaussiana) con media=0 y std=1
#   Valores típicamente entre -3 y 3 (99.7% de probabilidad).
#
# USO TÍPICO:
#   - rand(): Probabilidades, porcentajes, datos acotados
#   - randn(): Inicialización de pesos, ruido, datos simulados

print("\n5. torch.rand() y torch.randn() - Tensores aleatorios")
print("-" * 50)

# Semilla para reproducibilidad
torch.manual_seed(42)

uniform = torch.rand(2, 3)
normal = torch.randn(2, 3)

print(f"Uniforme [0,1):\n{uniform}")
print(f"\nNormal (μ=0, σ=1):\n{normal}")

# Para otras distribuciones:
# torch.randint(low, high, size): Enteros uniformes
# torch.normal(mean, std): Normal con parámetros específicos
enteros = torch.randint(0, 10, (3, 3))
print(f"\nEnteros aleatorios [0,10):\n{enteros}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# torch.arange(start, end, step) / torch.linspace(start, end, steps)
# ─────────────────────────────────────────────────────────────────────
# torch.arange():
#   Secuencia con paso fijo (como range() de Python).
#   El valor 'end' NO está incluido.
#
# torch.linspace():
#   Secuencia con número fijo de puntos equidistantes.
#   Ambos extremos ESTÁN incluidos.

print("\n6. torch.arange() y torch.linspace() - Secuencias")
print("-" * 50)

# arange: [start, end) con paso
seq1 = torch.arange(0, 10, 2)  # 0, 2, 4, 6, 8
print(f"arange(0, 10, 2):   {seq1}")

seq2 = torch.arange(5)  # Equivale a arange(0, 5, 1)
print(f"arange(5):          {seq2}")

# linspace: start y end incluidos, n puntos
seq3 = torch.linspace(0, 1, 5)  # 5 puntos entre 0 y 1
print(f"linspace(0, 1, 5):  {seq3}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# torch.eye(n) / torch.full(size, fill_value)
# ─────────────────────────────────────────────────────────────────────
# torch.eye():
#   Matriz identidad (unos en diagonal, ceros fuera).
#
# torch.full():
#   Tensor lleno de un valor específico.

print("\n7. torch.eye() y torch.full() - Tensores especiales")
print("-" * 50)

identidad = torch.eye(3)
print(f"Matriz identidad 3x3:\n{identidad}")

constante = torch.full((2, 4), fill_value=3.14)
print(f"\nTensor 2x4 lleno de 3.14:\n{constante}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# torch.zeros_like(input) / torch.ones_like(input) / torch.rand_like(input)
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Crea un tensor con la MISMA FORMA y DTYPE que el tensor de entrada.
#   Muy útil para crear tensores auxiliares compatibles.

print("\n8. Funciones *_like() - Copiar forma de otro tensor")
print("-" * 50)

original = torch.randn(2, 3)
print(f"Original:           shape {original.shape}, dtype {original.dtype}")

como_ceros = torch.zeros_like(original)
como_unos = torch.ones_like(original)
como_random = torch.randn_like(original)

print(f"zeros_like:         shape {como_ceros.shape}")
print(f"ones_like:          shape {como_unos.shape}")
print(f"randn_like:         shape {como_random.shape}")

### 1.2 Atributos de Tensores

Cada tensor tiene varios atributos importantes que debemos conocer:

In [ ]:
print("="*70)
print("ATRIBUTOS DE TENSORES")
print("="*70)

# Crear un tensor de ejemplo
t = torch.randn(3, 4, 5)

print(f"\nTensor de ejemplo: torch.randn(3, 4, 5)")
print("-" * 50)

# ─────────────────────────────────────────────────────────────────────
# tensor.shape / tensor.size()
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Devuelve las dimensiones del tensor.
#   .shape es un atributo, .size() es un método (equivalentes).

print(f"\n.shape:             {t.shape}")
print(f".size():            {t.size()}")
print(f".size(0):           {t.size(0)}  (primera dimensión)")
print(f".size(1):           {t.size(1)}  (segunda dimensión)")

# ─────────────────────────────────────────────────────────────────────
# tensor.dtype
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Tipo de dato almacenado en el tensor.
#
# TIPOS COMUNES:
#   torch.float32 (torch.float): 32-bit flotante, DEFAULT para redes
#   torch.float64 (torch.double): 64-bit flotante, alta precisión
#   torch.float16 (torch.half): 16-bit flotante, para GPUs modernas
#   torch.int64 (torch.long): 64-bit entero, para índices y etiquetas
#   torch.int32 (torch.int): 32-bit entero
#   torch.bool: Booleanos

print(f"\n.dtype:             {t.dtype}")

# ─────────────────────────────────────────────────────────────────────
# tensor.device
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Dispositivo donde reside el tensor (CPU o GPU).
#
# VALORES:
#   device(type='cpu'): En memoria RAM
#   device(type='cuda', index=0): En GPU 0

print(f".device:            {t.device}")

# ─────────────────────────────────────────────────────────────────────
# tensor.requires_grad
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Si True, PyTorch rastreará todas las operaciones para calcular
#   gradientes. Necesario para parámetros entrenables.

print(f".requires_grad:     {t.requires_grad}")

# ─────────────────────────────────────────────────────────────────────
# tensor.ndim / tensor.dim()
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Número de dimensiones del tensor.

print(f"\n.ndim:              {t.ndim}  (número de dimensiones)")

# ─────────────────────────────────────────────────────────────────────
# tensor.numel()
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Número total de elementos en el tensor.
#   Producto de todas las dimensiones.

print(f".numel():           {t.numel()}  (3×4×5 = 60 elementos)")

### 1.3 Dispositivos: CPU vs GPU

Una de las grandes ventajas de PyTorch es la facilidad para mover tensores entre CPU y GPU.

In [ ]:
print("="*70)
print("GESTIÓN DE DISPOSITIVOS (CPU/GPU)")
print("="*70)

# ─────────────────────────────────────────────────────────────────────
# tensor.to(device) / tensor.cuda() / tensor.cpu()
# ─────────────────────────────────────────────────────────────────────
# tensor.to(device):
#   Método flexible para mover tensores a cualquier dispositivo.
#   También puede cambiar dtype.
#
# tensor.cuda():
#   Atajo para mover a GPU (equivale a .to('cuda'))
#
# tensor.cpu():
#   Atajo para mover a CPU (equivale a .to('cpu'))
#
# NOTA IMPORTANTE:
#   Estos métodos RETORNAN un nuevo tensor.
#   El tensor original NO se modifica.

# Patrón recomendado para código portable
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nDispositivo seleccionado: {device}")

# Crear tensor en CPU
t_cpu = torch.randn(3, 3)
print(f"\nTensor original:    {t_cpu.device}")

# Mover al dispositivo óptimo
t_device = t_cpu.to(device)
print(f"Después de .to():   {t_device.device}")

# Crear directamente en el dispositivo
t_directo = torch.randn(3, 3, device=device)
print(f"Creado en device:   {t_directo.device}")

# ADVERTENCIA: Las operaciones entre tensores en diferentes dispositivos fallan
print("\n⚠️  IMPORTANTE: Los tensores deben estar en el mismo dispositivo")
print("   para realizar operaciones entre ellos.")

---

## 2. Operaciones con Tensores

PyTorch soporta una amplia gama de operaciones matemáticas. Todas las operaciones están optimizadas y pueden ejecutarse en GPU.

In [ ]:
print("="*70)
print("OPERACIONES ELEMENTO A ELEMENTO (ELEMENTWISE)")
print("="*70)

# Estas operaciones se aplican a cada elemento individualmente.
# Los tensores deben tener la misma forma o ser compatibles via broadcasting.

a = torch.tensor([1.0, 2.0, 3.0, 4.0])
b = torch.tensor([2.0, 2.0, 2.0, 2.0])

print(f"\na = {a}")
print(f"b = {b}")
print("-" * 50)

# ─────────────────────────────────────────────────────────────────────
# Operadores aritméticos: +, -, *, /, **, //
# ─────────────────────────────────────────────────────────────────────
# Equivalentes a torch.add(), torch.sub(), torch.mul(), torch.div(), torch.pow()

print(f"\na + b = {a + b}")
print(f"a - b = {a - b}")
print(f"a * b = {a * b}         (producto elemento a elemento)")
print(f"a / b = {a / b}         (división elemento a elemento)")
print(f"a ** 2 = {a ** 2}       (potencia elemento a elemento)")
print(f"a // b = {a // b}       (división entera)")
print(f"a % b = {a % 3}          (módulo)")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Funciones matemáticas elemento a elemento
# ─────────────────────────────────────────────────────────────────────

print("\nFUNCIONES MATEMÁTICAS")
print("-" * 50)

x = torch.tensor([0.0, 1.0, 2.0, 3.0])
print(f"x = {x}")
print()

# torch.sqrt(x): Raíz cuadrada
print(f"torch.sqrt(x):      {torch.sqrt(x)}")

# torch.exp(x): Exponencial (e^x)
print(f"torch.exp(x):       {torch.exp(x)}")

# torch.log(x): Logaritmo natural
print(f"torch.log(x+1):     {torch.log(x + 1)}  (sumamos 1 para evitar log(0))")

# torch.abs(x): Valor absoluto
y = torch.tensor([-1.0, -2.0, 3.0])
print(f"torch.abs({y}): {torch.abs(y)}")

# torch.clamp(x, min, max): Recortar valores a un rango
z = torch.tensor([0.5, 1.5, 2.5, 3.5])
print(f"torch.clamp({z}, 1, 3): {torch.clamp(z, 1, 3)}")

# Funciones trigonométricas
angles = torch.tensor([0, np.pi/4, np.pi/2])
print(f"\ntorch.sin():        {torch.sin(angles)}")
print(f"torch.cos():        {torch.cos(angles)}")

In [ ]:
print("="*70)
print("OPERACIONES DE REDUCCIÓN")
print("="*70)
print()

# Las operaciones de reducción colapsan una o más dimensiones,
# produciendo un tensor con menos elementos.

t = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])

print(f"Tensor 2x3:\n{t}")
print("-" * 50)

# ─────────────────────────────────────────────────────────────────────
# tensor.sum(dim=None, keepdim=False)
# ─────────────────────────────────────────────────────────────────────
# PARÁMETROS:
#   dim: Dimensión(es) a reducir. None suma todo.
#   keepdim: Si True, mantiene la dimensión reducida (con tamaño 1).

print(f"\n.sum():             {t.sum()}          (suma todo)")
print(f".sum(dim=0):        {t.sum(dim=0)}    (suma columnas → 1 fila)")
print(f".sum(dim=1):        {t.sum(dim=1)}          (suma filas → 1 columna)")

# ─────────────────────────────────────────────────────────────────────
# Otras funciones de reducción: mean, max, min, prod, std, var
# ─────────────────────────────────────────────────────────────────────

print(f"\n.mean():            {t.mean()}")
print(f".max():             {t.max()}")
print(f".min():             {t.min()}")
print(f".prod():            {t.prod()}  (producto de todos)")
print(f".std():             {t.std():.4f}  (desviación estándar)")

# ─────────────────────────────────────────────────────────────────────
# tensor.argmax(dim=None) / tensor.argmin(dim=None)
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Retorna el ÍNDICE del máximo/mínimo, no el valor.
#   Muy usado para obtener predicciones de clasificación.

print(f"\n.argmax():          {t.argmax()}  (índice del máximo en tensor aplanado)")
print(f".argmax(dim=1):     {t.argmax(dim=1)}  (índice del máximo por fila)")

In [ ]:
print("="*70)
print("OPERACIONES MATRICIALES")
print("="*70)
print()

# ─────────────────────────────────────────────────────────────────────
# torch.matmul(a, b) / a @ b
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Multiplicación matricial (producto punto generalizado).
#   El operador @ es equivalente.
#
# REGLA DE DIMENSIONES:
#   Para A(m×n) @ B(n×p) = C(m×p)
#   La última dimensión de A debe coincidir con la penúltima de B.

A = torch.randn(2, 3)  # 2 filas, 3 columnas
B = torch.randn(3, 4)  # 3 filas, 4 columnas

C = torch.matmul(A, B)  # También: A @ B

print(f"A: {A.shape}")
print(f"B: {B.shape}")
print(f"A @ B: {C.shape}  (2×3 @ 3×4 = 2×4)")

# ─────────────────────────────────────────────────────────────────────
# Batched matrix multiplication
# ─────────────────────────────────────────────────────────────────────
# matmul también funciona con batches de matrices.
# Las dimensiones extra se tratan como batch.

batch_A = torch.randn(32, 2, 3)  # 32 matrices de 2×3
batch_B = torch.randn(32, 3, 4)  # 32 matrices de 3×4
batch_C = batch_A @ batch_B

print(f"\nBatch de 32 multiplicaciones:")
print(f"batch_A: {batch_A.shape}")
print(f"batch_B: {batch_B.shape}")
print(f"batch_A @ batch_B: {batch_C.shape}")

In [ ]:
print("="*70)
print("MANIPULACIÓN DE FORMA (SHAPE)")
print("="*70)
print()

t = torch.arange(12)  # [0, 1, 2, ..., 11]
print(f"Tensor original: {t}")
print(f"Shape: {t.shape}")
print("-" * 50)

# ─────────────────────────────────────────────────────────────────────
# tensor.reshape(shape) / tensor.view(shape)
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Cambia la forma del tensor sin cambiar los datos.
#   El número total de elementos debe ser el mismo.
#
# DIFERENCIA:
#   - view(): Requiere tensor contiguo en memoria. Más eficiente.
#   - reshape(): Funciona siempre (puede hacer copia si es necesario).
#
# USAR -1 PARA INFERIR:
#   Una dimensión puede ser -1, y PyTorch la calcula automáticamente.

print(f"\n.reshape(3, 4):")
print(t.reshape(3, 4))

print(f"\n.reshape(2, 6):")
print(t.reshape(2, 6))

print(f"\n.reshape(2, -1):  (-1 se infiere como 6)")
print(t.reshape(2, -1))

print(f"\n.reshape(-1, 3):  (-1 se infiere como 4)")
print(t.reshape(-1, 3))

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# tensor.squeeze() / tensor.unsqueeze(dim)
# ─────────────────────────────────────────────────────────────────────
# squeeze(): Elimina TODAS las dimensiones de tamaño 1.
# squeeze(dim): Elimina la dimensión específica SI tiene tamaño 1.
# unsqueeze(dim): Añade una dimensión de tamaño 1 en la posición indicada.
#
# MUY USADOS para ajustar dimensiones en redes neuronales.

print("\nSQUEEZE y UNSQUEEZE")
print("-" * 50)

t = torch.randn(1, 3, 1, 4)
print(f"Original:           {t.shape}")
print(f".squeeze():         {t.squeeze().shape}  (elimina dims de tamaño 1)")
print(f".squeeze(0):        {t.squeeze(0).shape}  (elimina solo dim 0)")
print(f".squeeze(2):        {t.squeeze(2).shape}  (elimina solo dim 2)")

t2 = torch.randn(3, 4)
print(f"\nOriginal:           {t2.shape}")
print(f".unsqueeze(0):      {t2.unsqueeze(0).shape}  (añade dim al inicio)")
print(f".unsqueeze(1):      {t2.unsqueeze(1).shape}  (añade dim en posición 1)")
print(f".unsqueeze(-1):     {t2.unsqueeze(-1).shape}  (añade dim al final)")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# tensor.transpose(dim0, dim1) / tensor.permute(*dims)
# ─────────────────────────────────────────────────────────────────────
# transpose(): Intercambia DOS dimensiones.
# permute(): Reordena TODAS las dimensiones.
#
# MUY USADO para cambiar entre formatos de imagen:
#   (batch, height, width, channels) ↔ (batch, channels, height, width)

print("\nTRANSPOSE y PERMUTE")
print("-" * 50)

t = torch.randn(2, 3, 4)
print(f"Original:               {t.shape}")
print(f".transpose(0, 1):       {t.transpose(0, 1).shape}")
print(f".transpose(1, 2):       {t.transpose(1, 2).shape}")

# Permute para reordenar todas las dimensiones
print(f".permute(2, 0, 1):      {t.permute(2, 0, 1).shape}")

# Ejemplo práctico: imagen de (H, W, C) a (C, H, W)
img_hwc = torch.randn(224, 224, 3)  # Height, Width, Channels (formato PIL)
img_chw = img_hwc.permute(2, 0, 1)  # Channels, Height, Width (formato PyTorch)
print(f"\nConversión de imagen:")
print(f"  (H, W, C): {img_hwc.shape} → (C, H, W): {img_chw.shape}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# tensor.flatten(start_dim=0, end_dim=-1)
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Aplana dimensiones contiguas en una sola.
#
# MUY USADO antes de capas fully connected en CNNs.

print("\nFLATTEN")
print("-" * 50)

# Simular salida de conv: (batch, channels, height, width)
conv_output = torch.randn(32, 64, 7, 7)
print(f"Salida conv:        {conv_output.shape}")

# Aplanar todo excepto batch (dim 0)
flattened = conv_output.flatten(start_dim=1)
print(f"Después de flatten: {flattened.shape}  (64×7×7 = 3136)")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# torch.cat(tensors, dim=0) / torch.stack(tensors, dim=0)
# ─────────────────────────────────────────────────────────────────────
# cat(): Concatena tensores a lo largo de una dimensión EXISTENTE.
# stack(): Apila tensores creando una dimensión NUEVA.

print("\nCONCATENACIÓN Y APILAMIENTO")
print("-" * 50)

t1 = torch.randn(2, 3)
t2 = torch.randn(2, 3)
t3 = torch.randn(2, 3)

print(f"Tres tensores de shape {t1.shape}")

# cat: concatena, no crea nueva dimensión
cat_dim0 = torch.cat([t1, t2, t3], dim=0)
cat_dim1 = torch.cat([t1, t2, t3], dim=1)
print(f"\ntorch.cat dim=0:    {cat_dim0.shape}  (2+2+2=6 filas)")
print(f"torch.cat dim=1:    {cat_dim1.shape}  (3+3+3=9 columnas)")

# stack: crea nueva dimensión
stacked = torch.stack([t1, t2, t3], dim=0)
print(f"\ntorch.stack dim=0:  {stacked.shape}  (nueva dim de 3 al inicio)")
print(f"torch.stack dim=1:  {torch.stack([t1, t2, t3], dim=1).shape}")

---

## 3. Autograd: Diferenciación Automática

**Autograd** es el motor de diferenciación automática de PyTorch. Es lo que hace posible el entrenamiento de redes neuronales mediante backpropagation.

In [ ]:
print("="*70)
print("AUTOGRAD: DIFERENCIACIÓN AUTOMÁTICA")
print("="*70)
print()

# ─────────────────────────────────────────────────────────────────────
# requires_grad=True
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Cuando un tensor tiene requires_grad=True, PyTorch rastreará
#   todas las operaciones realizadas sobre él para poder calcular
#   gradientes después.
#
# REGLA:
#   Si CUALQUIER tensor de entrada tiene requires_grad=True,
#   el tensor de salida también lo tendrá.

# Crear tensores con seguimiento de gradientes
x = torch.tensor([2.0], requires_grad=True)  # Variable de entrada
w = torch.tensor([3.0], requires_grad=True)  # Peso
b = torch.tensor([1.0], requires_grad=True)  # Bias

print("Variables creadas con requires_grad=True:")
print(f"  x = {x.item():.1f}, w = {w.item():.1f}, b = {b.item():.1f}")
print()

# ─────────────────────────────────────────────────────────────────────
# Forward pass: construir grafo computacional
# ─────────────────────────────────────────────────────────────────────

# Cada operación añade un nodo al grafo
y = w * x + b   # y = 3*2 + 1 = 7
L = y ** 2      # L = 7² = 49 (función de pérdida simple)

print("Forward pass:")
print(f"  y = w*x + b = {w.item():.1f}*{x.item():.1f} + {b.item():.1f} = {y.item():.1f}")
print(f"  L = y² = {y.item():.1f}² = {L.item():.1f}")
print()

# ─────────────────────────────────────────────────────────────────────
# tensor.backward()
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Calcula los gradientes de todos los tensores con requires_grad=True
#   que contribuyeron al valor del tensor actual.
#
# REQUISITO:
#   Solo se puede llamar en tensores escalares (un solo elemento).
#   Para tensores no escalares, se necesita especificar un vector gradiente.

L.backward()  # Calcular todos los gradientes

print("Después de L.backward():")
print()

# ─────────────────────────────────────────────────────────────────────
# tensor.grad
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Contiene el gradiente calculado por backward().
#   ∂L/∂tensor

# Verificación matemática:
# L = y² = (wx + b)²
# ∂L/∂y = 2y = 2*7 = 14
# ∂L/∂w = ∂L/∂y * ∂y/∂w = 2y * x = 14 * 2 = 28
# ∂L/∂x = ∂L/∂y * ∂y/∂x = 2y * w = 14 * 3 = 42
# ∂L/∂b = ∂L/∂y * ∂y/∂b = 2y * 1 = 14

print("Gradientes calculados (∂L/∂variable):")
print(f"  w.grad = ∂L/∂w = 2y × x = 2×{y.item():.1f}×{x.item():.1f} = {w.grad.item():.1f}")
print(f"  x.grad = ∂L/∂x = 2y × w = 2×{y.item():.1f}×{w.item():.1f} = {x.grad.item():.1f}")
print(f"  b.grad = ∂L/∂b = 2y × 1 = 2×{y.item():.1f} = {b.grad.item():.1f}")

In [ ]:
print("="*70)
print("CONTROL DE GRADIENTES")
print("="*70)
print()

x = torch.randn(3, requires_grad=True)

# ─────────────────────────────────────────────────────────────────────
# torch.no_grad()
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Context manager que desactiva el seguimiento de gradientes.
#   Las operaciones dentro del bloque no se añaden al grafo.
#
# USO:
#   - Durante evaluación/inferencia (ahorra memoria)
#   - Al actualizar pesos manualmente

print("1. torch.no_grad() - Desactivar seguimiento")
print("-" * 50)

print(f"x.requires_grad: {x.requires_grad}")

with torch.no_grad():
    y = x * 2
    print(f"Dentro de no_grad: y.requires_grad = {y.requires_grad}")

# ─────────────────────────────────────────────────────────────────────
# tensor.detach()
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Retorna un nuevo tensor "desconectado" del grafo computacional.
#   El nuevo tensor comparte datos pero no tiene historial.

print(f"\n2. tensor.detach() - Desconectar del grafo")
print("-" * 50)

z = x.detach()
print(f"x.requires_grad: {x.requires_grad}")
print(f"x.detach().requires_grad: {z.requires_grad}")

# ─────────────────────────────────────────────────────────────────────
# Limpieza de gradientes
# ─────────────────────────────────────────────────────────────────────
# IMPORTANTE: Los gradientes se ACUMULAN en cada backward().
# Antes de cada paso de optimización, deben limpiarse.

print(f"\n3. Limpieza de gradientes")
print("-" * 50)

w = torch.tensor([1.0], requires_grad=True)

# Primer backward
(w * 2).sum().backward()
print(f"Después de 1er backward: w.grad = {w.grad}")

# Segundo backward SIN limpiar (los gradientes se suman)
(w * 2).sum().backward()
print(f"Después de 2do backward: w.grad = {w.grad}  ← ¡Se acumuló!")

# Limpiar gradientes
w.grad.zero_()  # In-place con _
print(f"Después de .grad.zero_(): w.grad = {w.grad}")

# O establecer a None
w.grad = None
print(f"Después de grad = None: w.grad = {w.grad}")

---

## 4. torch.nn: Construcción de Redes

El módulo `torch.nn` proporciona los bloques de construcción para crear redes neuronales.

In [ ]:
print("="*70)
print("CAPAS BÁSICAS DE torch.nn")
print("="*70)
print()

# ─────────────────────────────────────────────────────────────────────
# nn.Linear(in_features, out_features, bias=True)
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Capa fully connected (densa). Aplica transformación lineal:
#   y = x @ W.T + b
#
# PARÁMETROS:
#   in_features: Tamaño de cada entrada
#   out_features: Tamaño de cada salida
#   bias: Si incluir término de sesgo (default: True)
#
# PARÁMETROS APRENDIBLES:
#   weight: (out_features, in_features)
#   bias: (out_features,) si bias=True

print("1. nn.Linear - Capa Fully Connected")
print("-" * 50)

linear = nn.Linear(in_features=10, out_features=5)

print(f"Entrada esperada:   (*, 10)")
print(f"Salida:             (*, 5)")
print(f"Peso:               {linear.weight.shape}")
print(f"Bias:               {linear.bias.shape}")
print(f"Parámetros totales: {10*5 + 5} = 55")

# Ejemplo de uso
x = torch.randn(32, 10)  # Batch de 32, 10 features
y = linear(x)
print(f"\nEjemplo: {x.shape} → {y.shape}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0)
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Convolución 2D. Fundamental para procesamiento de imágenes.
#   Aplica filtros que detectan patrones locales.
#
# PARÁMETROS:
#   in_channels: Canales de entrada (1 para grayscale, 3 para RGB)
#   out_channels: Número de filtros (canales de salida)
#   kernel_size: Tamaño del filtro (3 significa 3x3)
#   stride: Paso del filtro (default: 1)
#   padding: Píxeles añadidos al borde (default: 0)
#
# FÓRMULA TAMAÑO SALIDA:
#   H_out = (H_in + 2*padding - kernel_size) / stride + 1

print("\n2. nn.Conv2d - Convolución 2D")
print("-" * 50)

conv = nn.Conv2d(
    in_channels=3,      # RGB
    out_channels=16,    # 16 filtros
    kernel_size=3,      # Filtros 3x3
    stride=1,
    padding=1           # Mantiene tamaño
)

print(f"Configuración: 3 → 16 canales, kernel 3x3, padding 1")
print(f"Peso:          {conv.weight.shape}  (out, in, H, W)")
print(f"Parámetros:    {conv.weight.numel() + conv.bias.numel()}")

# Ejemplo
img = torch.randn(32, 3, 64, 64)  # Batch de 32 imágenes RGB 64x64
out = conv(img)
print(f"\nEjemplo: {img.shape} → {out.shape}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# nn.MaxPool2d(kernel_size, stride=None, padding=0)
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Reduce dimensiones espaciales tomando el máximo en cada región.
#   No tiene parámetros aprendibles.
#
# PROPÓSITO:
#   - Reducir dimensionalidad
#   - Introducir invarianza a pequeñas traslaciones
#   - Reducir overfitting

print("\n3. nn.MaxPool2d - Max Pooling")
print("-" * 50)

pool = nn.MaxPool2d(kernel_size=2, stride=2)

img = torch.randn(1, 16, 64, 64)
out = pool(img)
print(f"Entrada:  {img.shape}")
print(f"Salida:   {out.shape}  (64/2 = 32)")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Funciones de Activación
# ─────────────────────────────────────────────────────────────────────
# Disponibles como clases (nn.ReLU) o funciones (F.relu).
# Las clases son útiles en nn.Sequential.
# Las funciones son útiles en forward() personalizado.

print("\n4. Funciones de Activación")
print("-" * 50)

activations = {
    'nn.ReLU()': nn.ReLU(),
    'nn.Sigmoid()': nn.Sigmoid(),
    'nn.Tanh()': nn.Tanh(),
    'nn.LeakyReLU(0.1)': nn.LeakyReLU(0.1),
    'nn.GELU()': nn.GELU(),
    'nn.Softmax(dim=1)': nn.Softmax(dim=1),
}

x = torch.randn(2, 5)
print(f"Entrada:\n{x}")
print()

for name, act in activations.items():
    y = act(x)
    print(f"{name:<22} min={y.min().item():>7.4f}, max={y.max().item():>7.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Capas de Regularización
# ─────────────────────────────────────────────────────────────────────

print("\n5. Capas de Regularización")
print("-" * 50)

# nn.Dropout(p)
# Durante entrenamiento: pone a 0 elementos con probabilidad p,
# y escala los demás por 1/(1-p).
# Durante evaluación: no hace nada.
print("nn.Dropout(p=0.5):")
print("  - Training: 50% de activaciones → 0")
print("  - Eval: No hace nada")

dropout = nn.Dropout(p=0.5)
x = torch.ones(1, 10)

dropout.train()  # Modo entrenamiento
print(f"  Train mode: {dropout(x)}")

dropout.eval()   # Modo evaluación
print(f"  Eval mode:  {dropout(x)}")

# nn.BatchNorm1d / nn.BatchNorm2d
# Normaliza las activaciones a lo largo del batch.
# Acelera entrenamiento y actúa como regularización.
print("\nnn.BatchNorm2d(num_features):")
print("  Normaliza cada canal a media=0, std=1")
print("  Tiene parámetros aprendibles: gamma (escala) y beta (sesgo)")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# nn.Module: La clase base
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Clase base para todos los modelos en PyTorch.
#   Subclasificar nn.Module para crear modelos propios.
#
# MÉTODOS OBLIGATORIOS:
#   __init__(self): Definir capas como atributos
#   forward(self, x): Definir flujo de datos
#
# MÉTODOS ÚTILES:
#   .parameters(): Iterador sobre parámetros aprendibles
#   .train() / .eval(): Cambiar modo (afecta Dropout, BatchNorm)
#   .to(device): Mover modelo a CPU/GPU
#   .state_dict(): Diccionario con pesos para guardar/cargar

print("="*70)
print("CREACIÓN DE MODELOS PERSONALIZADOS")
print("="*70)
print()

class MLP(nn.Module):
    """
    Perceptrón Multicapa personalizado.
    
    Esta es la forma estándar de definir modelos en PyTorch.
    """
    
    def __init__(self, input_size, hidden_size, output_size):
        """
        Inicializar capas.
        
        IMPORTANTE: Llamar super().__init__() primero.
        """
        super().__init__()  # ¡Obligatorio!
        
        # Definir capas como atributos
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        """
        Definir el flujo de datos.
        
        Este método se llama cuando hacemos: output = modelo(input)
        """
        x = self.fc1(x)    # Capa lineal 1
        x = self.relu(x)    # Activación
        x = self.fc2(x)    # Capa lineal 2
        return x

# Crear instancia
modelo = MLP(input_size=784, hidden_size=256, output_size=10)
print(modelo)

# Contar parámetros
total_params = sum(p.numel() for p in modelo.parameters())
print(f"\nParámetros totales: {total_params:,}")

# Forward pass
x = torch.randn(32, 784)
out = modelo(x)
print(f"Forward pass: {x.shape} → {out.shape}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# nn.Sequential: Forma rápida de apilar capas
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Contenedor que ejecuta módulos en orden secuencial.
#   Útil para arquitecturas simples sin bifurcaciones.

print("\nUSANDO nn.Sequential")
print("-" * 50)

modelo_seq = nn.Sequential(
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 10)
)

print(modelo_seq)

# Forward pass automático
x = torch.randn(32, 784)
out = modelo_seq(x)
print(f"\nForward: {x.shape} → {out.shape}")

---

## 5. torch.optim: Optimizadores

Los optimizadores actualizan los parámetros del modelo basándose en los gradientes calculados.

In [ ]:
print("="*70)
print("OPTIMIZADORES")
print("="*70)
print()

modelo = nn.Linear(10, 2)  # Modelo simple para ejemplos

# ─────────────────────────────────────────────────────────────────────
# torch.optim.SGD(params, lr, momentum=0, weight_decay=0)
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Stochastic Gradient Descent.
#   La forma más básica de optimización.
#
# ACTUALIZACIÓN:
#   v = momentum * v + gradient
#   param = param - lr * v
#
# PARÁMETROS:
#   lr: Learning rate (tasa de aprendizaje)
#   momentum: Factor de momento (0.9 es común)
#   weight_decay: Regularización L2

sgd = optim.SGD(modelo.parameters(), lr=0.01, momentum=0.9)
print("SGD(lr=0.01, momentum=0.9)")
print("  Simple y efectivo para CNNs")
print()

# ─────────────────────────────────────────────────────────────────────
# torch.optim.Adam(params, lr=0.001, betas=(0.9, 0.999), weight_decay=0)
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Adam = Adaptive Moment Estimation.
#   Combina momentum con learning rates adaptativos.
#   El optimizador más usado actualmente.
#
# PARÁMETROS:
#   lr: Learning rate inicial
#   betas: Coeficientes para promedios móviles (momento, cuadrados)

adam = optim.Adam(modelo.parameters(), lr=0.001)
print("Adam(lr=0.001)")
print("  Adaptativo, buen default para la mayoría de casos")
print()

# ─────────────────────────────────────────────────────────────────────
# torch.optim.AdamW(params, lr=0.001, weight_decay=0.01)
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Adam con "decoupled weight decay".
#   Implementación correcta de regularización L2 para Adam.
#   Preferido para Transformers y modelos grandes.

adamw = optim.AdamW(modelo.parameters(), lr=0.001, weight_decay=0.01)
print("AdamW(lr=0.001, weight_decay=0.01)")
print("  Adam con regularización L2 correcta")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Métodos del optimizador
# ─────────────────────────────────────────────────────────────────────

print("\nMÉTODOS DEL OPTIMIZADOR")
print("-" * 50)

modelo = nn.Linear(10, 2)
optimizer = optim.Adam(modelo.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

# Datos de ejemplo
x = torch.randn(32, 10)
y = torch.randn(32, 2)

print("Ciclo de optimización:")
print()

# 1. Forward pass
pred = modelo(x)
loss = loss_fn(pred, y)
print(f"1. Forward pass: loss = {loss.item():.4f}")

# 2. optimizer.zero_grad()
# DESCRIPCIÓN: Limpia los gradientes de todos los parámetros.
# IMPORTANTE: Llamar ANTES de backward() para evitar acumulación.
optimizer.zero_grad()
print("2. optimizer.zero_grad(): Limpiar gradientes")

# 3. loss.backward()
# Calcular gradientes
loss.backward()
print("3. loss.backward(): Calcular gradientes")

# 4. optimizer.step()
# DESCRIPCIÓN: Actualiza los parámetros usando los gradientes.
optimizer.step()
print("4. optimizer.step(): Actualizar parámetros")

# Verificar que los parámetros cambiaron
pred_new = modelo(x)
loss_new = loss_fn(pred_new, y)
print(f"\nNuevo loss: {loss_new.item():.4f} (debería ser menor)")

---

## 6. DataLoader: Gestión de Datos

El `DataLoader` gestiona la carga de datos en batches, shuffling, y paralelización.

In [ ]:
print("="*70)
print("DATALOADER Y DATASETS")
print("="*70)
print()

# ─────────────────────────────────────────────────────────────────────
# torch.utils.data.TensorDataset(*tensors)
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Dataset simple que envuelve tensores.
#   Cada muestra es una tupla de elementos en la misma posición.

print("1. TensorDataset - Dataset desde tensores")
print("-" * 50)

# Crear datos
X = torch.randn(1000, 10)  # 1000 muestras, 10 features
y = torch.randint(0, 3, (1000,))  # Etiquetas (3 clases)

dataset = TensorDataset(X, y)

print(f"Dataset size: {len(dataset)}")
print(f"Muestra 0: X shape {dataset[0][0].shape}, y = {dataset[0][1].item()}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# torch.utils.data.DataLoader(dataset, batch_size, shuffle, ...)
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Itera sobre un dataset en batches.
#   Gestiona shuffling, batching, y carga paralela.
#
# PARÁMETROS:
#   dataset: Un Dataset de PyTorch
#   batch_size: Muestras por batch
#   shuffle: Si mezclar en cada época (True para train, False para val/test)
#   num_workers: Procesos paralelos para cargar datos
#   drop_last: Si descartar último batch incompleto
#   pin_memory: Optimización para GPU (True si usas CUDA)

print("\n2. DataLoader - Iterador de batches")
print("-" * 50)

loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,       # Mezclar cada época
    num_workers=0,      # Usar 0 en notebooks
    drop_last=False
)

print(f"Dataset: {len(dataset)} muestras")
print(f"Batch size: 64")
print(f"Número de batches: {len(loader)} (⌈1000/64⌉ = 16)")
print()

# Iterar sobre batches
print("Iterando sobre los primeros 3 batches:")
for i, (batch_x, batch_y) in enumerate(loader):
    print(f"  Batch {i}: X shape {batch_x.shape}, y shape {batch_y.shape}")
    if i >= 2:
        print("  ...")
        break

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# torch.utils.data.Dataset (clase base)
# ─────────────────────────────────────────────────────────────────────
# DESCRIPCIÓN:
#   Clase base para datasets personalizados.
#   Subclasificar e implementar __len__ y __getitem__.
#
# MÉTODOS OBLIGATORIOS:
#   __len__(self): Retorna el número de muestras
#   __getitem__(self, idx): Retorna la muestra en posición idx

print("\n3. Dataset Personalizado")
print("-" * 50)

class MiDataset(Dataset):
    """
    Dataset personalizado de ejemplo.
    
    Útil cuando los datos no caben en memoria o necesitan
    procesamiento especial.
    """
    
    def __init__(self, data, labels, transform=None):
        """
        Inicializar el dataset.
        
        Args:
            data: Datos de entrada
            labels: Etiquetas
            transform: Transformaciones a aplicar (data augmentation)
        """
        self.data = data
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        """Retorna el número total de muestras."""
        return len(self.data)
    
    def __getitem__(self, idx):
        """
        Obtiene una muestra por índice.
        
        Aquí es donde se carga/procesa cada muestra.
        """
        x = self.data[idx]
        y = self.labels[idx]
        
        # Aplicar transformaciones si existen
        if self.transform:
            x = self.transform(x)
        
        return x, y

# Usar el dataset personalizado
mi_dataset = MiDataset(X, y)
mi_loader = DataLoader(mi_dataset, batch_size=32, shuffle=True)

print(f"Dataset personalizado: {len(mi_dataset)} muestras")
print(f"DataLoader: {len(mi_loader)} batches")

---

## 7. Ciclo de Entrenamiento Completo

Aquí combinamos todo lo aprendido en un ejemplo completo.

In [ ]:
from torchvision import datasets, transforms

print("="*70)
print("EJEMPLO COMPLETO: CLASIFICACIÓN MNIST")
print("="*70)
print()

# ============================================================
# 1. CONFIGURACIÓN
# ============================================================
BATCH_SIZE = 64
EPOCHS = 5
LR = 0.001
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Configuración:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning rate: {LR}")
print(f"  Device: {DEVICE}")

In [ ]:
# ============================================================
# 2. CARGAR DATOS
# ============================================================
print("\n" + "="*50)
print("CARGANDO DATOS")
print("="*50)

# Transformaciones
transform = transforms.Compose([
    transforms.ToTensor(),  # Convierte PIL Image a tensor y escala a [0, 1]
    transforms.Normalize((0.1307,), (0.3081,))  # Media y std de MNIST
])

# Descargar datasets
train_data = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST('./data', train=False, transform=transform)

# Crear DataLoaders
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_data)} imágenes, {len(train_loader)} batches")
print(f"Test: {len(test_data)} imágenes, {len(test_loader)} batches")

In [ ]:
# ============================================================
# 3. DEFINIR MODELO
# ============================================================
print("\n" + "="*50)
print("DEFINIENDO MODELO")
print("="*50)

class CNN(nn.Module):
    """CNN simple para MNIST."""
    
    def __init__(self):
        super().__init__()
        
        # Capas convolucionales
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)  # 28x28 → 28x28
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)  # 14x14 → 14x14
        self.pool = nn.MaxPool2d(2, 2)  # Reduce a la mitad
        
        # Capas fully connected
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        
        self.dropout = nn.Dropout(0.25)
    
    def forward(self, x):
        # Conv block 1: 28x28 → 14x14
        x = self.pool(F.relu(self.conv1(x)))
        
        # Conv block 2: 14x14 → 7x7
        x = self.pool(F.relu(self.conv2(x)))
        
        # Flatten
        x = x.view(-1, 64 * 7 * 7)
        
        # FC layers
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

modelo = CNN().to(DEVICE)
print(modelo)

# Contar parámetros
params = sum(p.numel() for p in modelo.parameters())
print(f"\nParámetros totales: {params:,}")

In [ ]:
# ============================================================
# 4. CONFIGURAR LOSS Y OPTIMIZADOR
# ============================================================

loss_fn = nn.CrossEntropyLoss()  # Para clasificación multiclase
optimizer = optim.Adam(modelo.parameters(), lr=LR)

print(f"Loss: CrossEntropyLoss")
print(f"Optimizer: Adam(lr={LR})")

In [ ]:
# ============================================================
# 5. FUNCIONES DE ENTRENAMIENTO Y EVALUACIÓN
# ============================================================

def train_epoch(model, loader, loss_fn, optimizer, device):
    """
    Entrena el modelo por una época.
    
    Retorna: (loss promedio, accuracy)
    """
    model.train()  # Modo entrenamiento (activa Dropout, BatchNorm)
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_x, batch_y in loader:
        # Mover datos al dispositivo
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        # Forward pass
        outputs = model(batch_x)
        loss = loss_fn(outputs, batch_y)
        
        # Backward pass
        optimizer.zero_grad()  # Limpiar gradientes
        loss.backward()        # Calcular gradientes
        optimizer.step()       # Actualizar pesos
        
        # Métricas
        total_loss += loss.item() * batch_x.size(0)
        _, predicted = outputs.max(1)
        total += batch_y.size(0)
        correct += predicted.eq(batch_y).sum().item()
    
    return total_loss / total, 100 * correct / total


def evaluate(model, loader, loss_fn, device):
    """
    Evalúa el modelo.
    
    Retorna: (loss promedio, accuracy)
    """
    model.eval()  # Modo evaluación (desactiva Dropout, etc.)
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():  # No calcular gradientes
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            
            outputs = model(batch_x)
            loss = loss_fn(outputs, batch_y)
            
            total_loss += loss.item() * batch_x.size(0)
            _, predicted = outputs.max(1)
            total += batch_y.size(0)
            correct += predicted.eq(batch_y).sum().item()
    
    return total_loss / total, 100 * correct / total

In [ ]:
# ============================================================
# 6. ENTRENAMIENTO
# ============================================================
print("\n" + "="*50)
print("ENTRENANDO")
print("="*50 + "\n")

historial = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

for epoch in range(EPOCHS):
    # Entrenar
    train_loss, train_acc = train_epoch(modelo, train_loader, loss_fn, optimizer, DEVICE)
    
    # Evaluar
    test_loss, test_acc = evaluate(modelo, test_loader, loss_fn, DEVICE)
    
    # Guardar historial
    historial['train_loss'].append(train_loss)
    historial['train_acc'].append(train_acc)
    historial['test_loss'].append(test_loss)
    historial['test_acc'].append(test_acc)
    
    # Imprimir progreso
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"  Train - Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%")
    print(f"  Test  - Loss: {test_loss:.4f}, Acc: {test_acc:.2f}%")

print("\n" + "="*50)
print(f"RESULTADO FINAL: {test_acc:.2f}% accuracy en test")
print("="*50)

In [ ]:
# ============================================================
# 7. VISUALIZAR RESULTADOS
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

epochs = range(1, EPOCHS + 1)

# Loss
axes[0].plot(epochs, historial['train_loss'], 'b-o', label='Train')
axes[0].plot(epochs, historial['test_loss'], 'r-o', label='Test')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].set_title('Curva de Pérdida')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs, historial['train_acc'], 'b-o', label='Train')
axes[1].plot(epochs, historial['test_acc'], 'r-o', label='Test')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Curva de Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 8. Referencia Rápida de Funciones

### Creación de Tensores

| Función | Descripción |
|---------|-------------|
| `torch.tensor(data)` | Crear tensor desde datos (copia) |
| `torch.from_numpy(arr)` | Desde NumPy (comparte memoria) |
| `torch.zeros(size)` | Tensor de ceros |
| `torch.ones(size)` | Tensor de unos |
| `torch.rand(size)` | Uniforme [0, 1) |
| `torch.randn(size)` | Normal (0, 1) |
| `torch.arange(start, end, step)` | Secuencia con paso |
| `torch.linspace(start, end, steps)` | N puntos equidistantes |
| `torch.eye(n)` | Matriz identidad |
| `torch.full(size, value)` | Tensor lleno de un valor |

### Operaciones de Forma

| Método | Descripción |
|--------|-------------|
| `.reshape(shape)` | Cambiar forma |
| `.view(shape)` | Cambiar forma (contiguo) |
| `.squeeze()` | Eliminar dims de tamaño 1 |
| `.unsqueeze(dim)` | Añadir dim de tamaño 1 |
| `.transpose(d1, d2)` | Intercambiar dos dims |
| `.permute(*dims)` | Reordenar todas las dims |
| `.flatten(start_dim)` | Aplanar dimensiones |
| `torch.cat(tensors, dim)` | Concatenar |
| `torch.stack(tensors, dim)` | Apilar (nueva dim) |

### Reducciones

| Método | Descripción |
|--------|-------------|
| `.sum(dim)` | Suma |
| `.mean(dim)` | Media |
| `.max(dim)` | Máximo |
| `.min(dim)` | Mínimo |
| `.argmax(dim)` | Índice del máximo |
| `.std(dim)` | Desviación estándar |

### Capas nn

| Capa | Descripción |
|------|-------------|
| `nn.Linear(in, out)` | Fully connected |
| `nn.Conv2d(in_ch, out_ch, kernel)` | Convolución 2D |
| `nn.MaxPool2d(kernel)` | Max pooling |
| `nn.BatchNorm2d(features)` | Batch normalization |
| `nn.Dropout(p)` | Dropout |
| `nn.ReLU()` | Activación ReLU |
| `nn.Sigmoid()` | Activación Sigmoid |
| `nn.Softmax(dim)` | Softmax |

---

**Siguiente notebook**: [DL_03_arquitecturas.ipynb](DL_03_arquitecturas.ipynb)